# Battlesnake: Avoiding Immediate Danger

Let's take a look at how we need to respond to the [`move` HTTP POST request](https://docs.battlesnake.com/api/webhooks#response-properties-2).

```json
{
  "move": "up",
  "shout": "Moving up!" // this bit is optional
}
```

**We need to respond with a direction.**

Let's take a look at [an example `move` request payload](https://docs.battlesnake.com/api/example-move).

**We get all of our hazards in terms of coordinates.**

To determine if a move (direction) intersects with a hazard (coordinate), we need to be able to convert between the two.

## Objective 1: get_up_coord

Write a function that will take a coordinate and give us back the coordinate of the "up" direction. For example:

```python
up_coord = get_up_coord(head_coord={"x": 1, "y": 1})
```

WAIT! Don't start writing the function yet! Let's follow the precepts of Test Driven Development (TDD). We're not going to subscribe completely to TDD principals, but we're going to us an important one here:

1. Don't write any code until you have a failing test case

This might seem counterintuitive, but it forces us to think about testability from the inception of any change. If we wrote a functioning solution that is difficult - or impossible - to test, we might spend the next few days writing a test case fixture that requires more lines of code than the feature itself! Writing the test case first forces us to write the feature in a testable way from the first commit.

Let's write our first test case!


## Test Cases

Test cases in Python have come a long way. A lot of the Stackoverflow answers and documentation you'll find on the web will refer to a library called `unittests`. `unittests` was the de-facto testing library for Python for a long time. These days, all the cool kids are using the `pytest` library. Pytest is a large, mature library that deserves a much longer course with a much more qualified teacher. We'll cover basic concepts in this course. As you apply TDD to other code bases, you'll need to dive deeper into the library for more of its rich capabilities.

The starter repo already had some of the structure for writing tests. You'll find a top-level [tests](./tests) directory with a [test_snake](./tests/test_snake.py) file containing a __very__ simple test case: `test_info`. Let's run your test suite. First, **is your virtual environment activated?**

```bash
pytest .
```

You should see something like this:

```
=========================================================================================== test session starts ===========================================================================================
platform darwin -- Python 3.11.9, pytest-8.3.5, pluggy-1.5.0
rootdir: /Users/zanebclark/GitHubProjects/battlesnake-starter
configfile: pyproject.toml
plugins: anyio-4.9.0
collected 1 item

tests/test_snake.py .                                                                                                                                                                               [100%]

============================================================================================ 1 passed in 0.00s ============================================================================================
```

When you run `pytest .`, the library initially "collects" test cases from the directory you targeted. We targeted `.`, the directory that the terminal is currently at. Within that directory, the library searches for files that begin with `test_`. Within those files, it then searches for functions that begin with `test_` or classes that begin with `Test`.

After collecting your test cases, it will execute them and display the results.

We'll write all of our tests in the [tests](./tests) directory.
- First, it's where a rational person would look for tests. #berational
- Second, it will force the test cases to use the code in the [src](./src) directory as a library. That's not important to our program, but it will save you some heartache in the future if you write packages or libraries that you expect to be able to import and use.

So that Pytest can collect our tests, every file containing a test should start with `test_`. For now, let's add to our existing [test_snake](./tests/test_snake.py) file.

In [ ]:
def test_get_up_coord():  # The function name starts with "test_"
    # Given: The initial conditions
    head_coord = {"x": 1, "y": 1}

    # When: The action being tested. In this case, executing a function
    up_coord = get_up_coord(head_coord=head_coord)

    # Then: The expected outcome. In this case, we expect the return value of that function to be an explicit value.
    assert up_coord == {"x": 1, "y": 2}


Let's run the test case. Congratulations! You have a failing test case. You may now write some code. First, we need to define the function:

In [ ]:
def get_up_coord(head_coord: dict[str, int]) -> dict[str, int]:
    pass


Then, let's import the function in our [test_snake](./tests/test_snake.py) file:

In [ ]:
from battlesnakes.snake import info, get_up_coord


Run the test again. Now, we're getting a different error. That's exciting! I often develop with a single objective in mind: get a different error.

This time, Pytest is telling us that the return value doesn't match the expected value. Since we have a failing test case, we can write more code.

In [ ]:
def get_up_coord(head_coord: dict[str, int]) -> dict[str, int]:
    return {"x": head_coord["x"], "y": head_coord["y"] + 1}


Now our test case passes! We might want to test multiple cases to ensure it behaves as we would expect.

In [ ]:
def test_get_up_coord_2():  # The function name starts with "test_"
    # Given: The initial conditions
    head_coord = {"x": 9, "y": 9}

    # When: The action being tested. In this case, executing a function
    up_coord = get_up_coord(head_coord=head_coord)

    # Then: The expected outcome. In this case, we expect the return value of that function to be an explicit value.
    assert up_coord == {"x": 9, "y": 10}


That works too! A lot of that code is repetitive. Let's __parameterize__ the test case so we can reuse most of the code.

In [ ]:
import pytest


@pytest.mark.parametrize(
    "test_input,expected",
    [
        ({"x": 1, "y": 1}, {"x": 1, "y": 2}),
        ({"x": 9, "y": 9}, {"x": 9, "y": 10}),
    ],
)
def test_get_up_coord_parameterized(test_input, expected):
    up_coord = get_up_coord(head_coord=test_input)
    assert up_coord == expected


This does the same thing as our previous two test cases combined! The test case names don't tell us much about what we're testing. Let's fix that:

In [ ]:
def name_coordinate_test_cases(value):
    return f"({value['x']}, {value['y']})"


@pytest.mark.parametrize(
    "test_input,expected",
    [
        ({"x": 1, "y": 1}, {"x": 1, "y": 2}),
        ({"x": 9, "y": 9}, {"x": 9, "y": 10}),
    ],
    ids=name_coordinate_test_cases,
)
def test_get_up_coord_parameterized(test_input, expected):
    up_coord = get_up_coord(head_coord=test_input)
    assert up_coord == expected


Test cases are a great way to ensure that the code works the way you would expect it to given ideal circumstances. This is often referred to as testing the "happy path". It is equally important to test how your code responds to the "unhappy path". For example, we're assuming that the `head_coord` is a dictionary with "x" and "y" keys. If that's not the case, do we handle it gracefully? Let's find out.

In [1]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    up_coord = get_up_coord(head_coord=head_coord)

    assert up_coord == {"x": 9, "y": 10}


Because this is a test case, we get a nice printout of the input variable value. If this happened in the wild, we would just see `KeyError: 'x'` and have no way to understand which coordinate was a problem. Let's fix that:

In [ ]:
def get_up_coord(head_coord: dict[str, int]) -> dict[str, int]:
    if "x" not in head_coord.keys() or "y" not in head_coord.keys():
        raise ValueError(f"head_coord must have both 'x' and 'y' keys: {head_coord}")

    return {"x": head_coord["x"], "y": head_coord["y"] + 1}


If we rerun our test case, we'll get a much more verbose error: `ValueError: head_coord must have both 'x' and 'y' keys: {'y': 9}`. How do we communicate to Pytest that we expect this exception to be raised?

In [ ]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    with pytest.raises(
            ValueError, match="head_coord must have both 'x' and 'y' keys: .*"
    ):
        get_up_coord(head_coord=head_coord)

Here, we're asserting that an Exception is raised of a specific type with a specific error message. This test case passes. Beware of test cases that pass immediately. Let's alter it a bit to make sure it fails when it should:

In [ ]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    with pytest.raises(ValueError, match="the wrong message"):
        get_up_coord(head_coord=head_coord)

That fails, complaining that the message of the exception doesn't match the regular expression that we supplied. Let's return the correct message:

In [ ]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    with pytest.raises(
            ValueError, match="head_coord must have both 'x' and 'y' keys: .*"
    ):
        get_up_coord(head_coord=head_coord)
